# Part G - Agentic RAG

In [1]:
from pymilvus import MilvusClient
from sentence_transformers import SentenceTransformer
import json
from pathlib import Path
import datetime
from dotenv import load_dotenv
from pydantic_ai import Agent, UsageLimits
from pydantic_ai.settings import ModelSettings
from pydantic_ai.run import AgentRunResult
import nest_asyncio
from uuid import uuid4
from dataclasses import dataclass

In [2]:
#########################
# Initial configuration
#########################

nest_asyncio.apply()  # To be able to run pydantic in jupyter notebooks preventing event loop issues
load_dotenv()
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

BASE_PATH = Path().resolve()
CHAT_DATA_PATH = BASE_PATH.joinpath("chat_data")
CHAT_DATA_PATH.mkdir(parents=True, exist_ok=True)
RETRIEVAL_HISTORY_FILE_PATH = CHAT_DATA_PATH.joinpath("retrieval_history.json")

DATA_FILE_PATH = BASE_PATH.joinpath("20251101_100000_results.json")
OUTPUT_FILE_PATH = BASE_PATH.joinpath(f"{timestamp}_llm_responses_agentic_rag_results.json")

DATABASE_NAME = "tesis_MCD_v1"
COLLECTION_NAME = "codigo_de_mineria_v1"
EMBEDDING_MODEL_NAME = "hiiamsid/sentence_similarity_spanish_es"  # https://huggingface.co/hiiamsid/sentence_similarity_spanish_es/tree/main
ANSWER_VECTOR_FIELD_NAME = "vector_contenido_texto" # database field that contains the embedding vector to perform the search.
OUTPUT_FIELDS = [
    "codigo_norma",
    "id_parte",
    "id_padre",
    "tipo_parte",
    "numero_parte",
    "titulo_parte",
    "ultimo_estado",
    "contenido_texto",
]

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

MODELS=[
    "openai:gpt-5-nano",
    "openai:gpt-5-mini",
    "openai:gpt-5",
]


#########################
# Helper functions
#########################

def get_db_client():
    print("get_db_client")
    client = MilvusClient(uri="http://localhost:19530", token="root:Milvus")
    client.use_database(DATABASE_NAME)
    return client

def load_retrieval_metadata():
    print("load_retrieval_metadata")
    if RETRIEVAL_HISTORY_FILE_PATH.exists():
        with open(RETRIEVAL_HISTORY_FILE_PATH, encoding="utf-8", mode="r") as f:
            data = json.load(f)
        return data
    return {}

def save_retrieval_metadata(chat_id, query, retrieved_documents):
    print("save_retrieval_metadata")
    data = load_retrieval_metadata()
    result = {
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "query": query,
        "retrieved_documents": retrieved_documents
    }

    if data.get(chat_id, None):
        data[chat_id] = data[chat_id].append(result)
    else:
        data[chat_id] = [result]

    with open(RETRIEVAL_HISTORY_FILE_PATH, encoding="utf-8", mode="w") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

def get_retrieval_history(chat_id:str) -> dict:
    print("get_retrieval_history")
    data = load_retrieval_metadata()
    return data[chat_id]

def get_document_matches(text: str, max_documents: int = 10) -> list[dict]:
    try:
        client = get_db_client()
        question_vector = embedding_model.encode(text)

        results = client.search(
            collection_name=COLLECTION_NAME,
            anns_field=ANSWER_VECTOR_FIELD_NAME,
            data=[question_vector],
            limit=5000,
            search_params={"metric_type": "COSINE"},
            output_fields=OUTPUT_FIELDS,
        )

    except Exception as e:
        return e
    finally:
        client.close()

    return [
        {"index": i, "distance": result["distance"], **result["entity"]}
        for i, result in enumerate(results[0], start=1)
    ][:max_documents]


def save_conversation_metadata(chat_id: str, agent:Agent, model_name:str, result: AgentRunResult):
    conversation_data = {
        "messages": json.loads(result.all_messages_json()),
        "metadata": {
            "model_settings": agent.model_settings,
            "model_version": result.response.model_name,
            "provider_name": result.response.provider_name,
            "timestamp": result.timestamp().isoformat(),
            "usage": {
                "tokens": {
                    "input_tokens": result.usage().input_tokens,
                    "output_tokens": result.usage().output_tokens,
                    "tokens_details": result.usage().details,
                },
                "cost": {
                    "input_price": float(result.response.cost().input_price),
                    "output_price": float(result.response.cost().output_price),
                    "total_price": float(result.response.cost().total_price),
                    "model_input_price_mtok": float(result.response.cost().model_price.input_mtok),
                    "model_output_price_mtok": float(result.response.cost().model_price.output_mtok),
                    "model_id": result.response.cost().model.id,
                    "model_name": result.response.cost().model.name,
                },
            },
        }
    }
    CHAT_DIR_PATH = CHAT_DATA_PATH.joinpath(chat_id)
    CHAT_DIR_PATH.mkdir(exist_ok=True)
    with open(CHAT_DIR_PATH.joinpath(f"{model_name}.json"), encoding="utf-8") as f:
        json.dump(conversation_data, f, ensure_ascii=False, indent=4)

In [3]:
#########################
# Instructions
# The `instructions` parameter. Check the docs for more details: https://ai.pydantic.dev/agents/#instructions
#########################

INSTRUCTIONS = {
    "law_specialist_agent": """
## Descripción:

Eres un asistente legal experto en derecho Argentino. Respondes preguntas legales basadas en la Constitución Nacional Argentina, Códigos, Leyes y jurisprudencia.

## Requisitos

* Analizas la pregunta del usuario, extraes los puntos importantes y utilizas la herramienta `retrieve_relevant_documents` una vez para obtener información de la base de datos legislativa.
* Analizas los documentos para generar la respuesta final.
* Siempre citas las fuentes legales consultadas para la elaboración de tu respuesta en formato APA.
* Si no sabes la respuesta y los documentos legales no son suficientes para crear una respuesta, di que no cuentas con información para responderla.
* Las respuestas deben ser claras, concisas y directas al punto.
* Solo retorna la respuesta. No agregues texto irrelevante.
* Respondes en español al menos que el usuario solicite otro idioma.
""",


    "retrieval_agent": """
Eres un especialista en búsqueda de información legislativa que:

* Recibes una consulta relacionada con cuestiones legales de derecho Argentino.
* La consulta puede dividirse en subconsultas para mejorar la recuperación.
* Realizas una búsqueda por cada subconsulta en una base de datos de vectores empleando la herramienta `cosine_distance_vector_search`.
* Utilizas la herramienta `evaluate_retrieval` para evaluar la calidad de la recuperación.
* Respondes con los resultados de la búsqueda.
""",


    "retrieval_quality_agent": """
## Descripción:

Eres un especialista en "retrieval augmented generation" encargado de evaluar la calidad de la recuperación.

* Comparas la consulta original con la información recuperada y decides si la información es suficiente para responder la consulta.
* Provees feedback para mejorar la consulta.
* El feedback debe ser conciso.
""",
}

#########################
# Limits
#########################

DOCUMENT_RETRIEVAL_LIMIT = UsageLimits(tool_calls_limit=3)
VECTOR_SEARCH_LIMIT = UsageLimits(tool_calls_limit=10)

#########################
# Dependencies
#########################


#########################
# Outputs
#########################
@dataclass
class RetrievalFinalOutput:
    retrieved_documents: list[dict]

@dataclass
class RetrievalEvaluation:
    feedback_text: str
    repeat_search: bool

@dataclass
class ExpertResponse:
    response_text: str

#########################
# Tools
#########################

# Agent tools
def retrieve_relevant_documents(question: str, chat_id:str) -> dict[str,]:
    """
    Retrieves documents to answer the user's questions.
    """
    result = retrieval_agent.run_sync(
        f"""Chat id: {chat_id}
        Please retrieve documents for this user query:
        > {question}
        """,
        usage_limits=VECTOR_SEARCH_LIMIT,
        output_type=RetrievalFinalOutput
    )
    save_conversation_metadata(chat_id, retrieval_agent, "retrieval_agent", result)

    return result.output

def evaluate_retrieval(chat_id:str, original_query:str, search_query:str, retrieved_data:list[dict]):
    """
    original_query: the user's original query.
    search_query: the query used by the retrieval agent to get the retrieved data.
    retrieved_data: resulting data from the retrieval operation, for example:
    ```json
        {
        "index": 1,
        "distance": 0.6904841065406799,
        "tipo_parte": "artículo",
        "numero_parte": 25,
        "titulo_parte": null,
        "ultimo_estado": "en vigor",
        "contenido_texto": "Some text content",
        "codigo_norma": "Código de Minería Argentino",
        "id_padre": "7305f9fee43e3553ad5e856cdf3c99ff",
        "id_parte": "8d21b964b0fa52b6b673df7724c42c55"
    },
    ```
    """
    result = retrieval_quality_agent.run_sync(
        f"""Chat id: {chat_id}
        Please evaluate the quality for the following case:
        User query: {original_query}
        Search query: {search_query}
        Search result: {retrieved_data}

        Do the results answer the question or should I search again? 
        """,
        usage_limits=DOCUMENT_RETRIEVAL_LIMIT,
        output_type=RetrievalEvaluation,
    )
    save_conversation_metadata(chat_id, retrieval_quality_agent, "retrieval_quality_agent", result)

    return result.output

# Code tools
def cosine_distance_vector_search(chat_id:str, query:str) -> list[dict]:
    result = get_document_matches(query, max_documents=5)
    save_retrieval_metadata(chat_id, query, result)
    return result

def get_documents_by_ids(id_list:list[str]) -> list[dict]:
    try:
        client = get_db_client()
        results = client.get(
            collection_name=COLLECTION_NAME, ids=id_list, output_fields=OUTPUT_FIELDS
        )
    except Exception as e:
        print(e)
    finally:
        client.close()

    return list(results)

#########################
# Agents
# Definition of agents to be used globally per chat session.
#########################
model_settings = ModelSettings(temperature=0.0)

law_specialist_agent = Agent(
    model=MODELS[2], 
    instructions=INSTRUCTIONS["law_specialist_agent"],
    tools=[retrieve_relevant_documents],
    model_settings=model_settings,
)
retrieval_agent = Agent(
    model=MODELS[2],
    instructions=INSTRUCTIONS["retrieval_agent"],
    tools=[cosine_distance_vector_search, evaluate_retrieval],
    model_settings=model_settings,
)
retrieval_quality_agent = Agent(
    model=MODELS[2], 
    instructions=INSTRUCTIONS["retrieval_quality_agent"],
    model_settings=model_settings,
)

#########################
# Main functions
#########################

def answer_user(question:str, chat_id:str) -> AgentRunResult:
    print("Generating answer")
    result = law_specialist_agent.run_sync(
        question,
        output_type=ExpertResponse,
    )
    print("Saving results")
    save_conversation_metadata(chat_id, law_specialist_agent, "law_specialist_agent", result)
    return result.output

In [4]:
"""
# Workflow description

1. User submits a question to the `Law Specialist Agent`
2. The `Law Specialist Agent` processes the question structuring it for the `Retrieval Agent`
3. Based on an Agent's input, the `Retrieval Agent` prepares query for the vector database and uses a tool to perform the query.
4. A `Retrieval Quality Agent` evaluates the output and decides if the returned data is sufficient to answer the query
    4.1. If not, the model prepares a short helpful feedback and instructs the `Retrieval Agent` to generate new queries based on the feedback.
    4.2. If it is, the retrieved data is passed to the `Law Specialist Agent` to generate the final answer.
    - LIMIT Sequence to 3 times.
    - The retrieval quality agent needs to have a memory tool to interact with previous retrievals
        - Get best matches based on the metric distance
        - Store ids of the retrieved documents.
5. The `Law Specialist Agent` returns the final answer to the user.

Tools:
    - Milvus
        - Retrieve documents by semantic search
        - Retrieve document by id
```

"""


with open(DATA_FILE_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

# for pregunta in data[:1]:
#     chat_id = uuid4().hex
#     answer = answer_user(pregunta["texto"], chat_id)

#     if "llm_responses_agentic_rag" not in pregunta:
#         pregunta["llm_responses_agentic_rag"] = []

#     pregunta["llm_responses_agentic_rag"].append(answer)

#     with open(OUTPUT_FILE_PATH, encoding="utf-8", mode="w") as f:
#         f.write(json.dumps(data, ensure_ascii=False, indent=4))

In [12]:
for pregunta in data:
    print(f"{pregunta['numero_pregunta']:02d}_{pregunta['numero_subpregunta'] if pregunta['numero_subpregunta'] else 0:02d}")

01_01
01_02
01_03
02_01
02_02
02_03
03_01
03_02
03_03
04_01
04_02
04_03
05_01
05_02
05_03
05_04
06_01
06_02
06_03
06_04
06_05
07_01
07_02
07_03
07_04
07_05
08_01
08_02
08_03
09_01
09_02
09_03
09_04
10_01
10_02
10_03
11_01
11_02
11_03
11_04
12_01
12_02
12_03
13_01
13_02
13_03
14_01
14_02
14_03
14_04
15_01
15_02
15_03
16_01
16_02
16_03
16_04
17_01
17_02
17_03
18_01
18_02
18_03
19_01
19_02
19_03
19_04
19_05
20_01
20_02
20_03
21_01
21_02
21_03
22_01
22_02
23_00
24_00
25_00


In [ ]:
RETRIEVAL_HISTORY_DIR_PATH = Path.cwd().resolve().joinpath("chat_data", "01_01", "retrieval_history")

def get_previous_search_results():
    results = []
    for document in RETRIEVAL_HISTORY_DIR_PATH.glob("*.json"):
        with open(document, mode="r", encoding="utf-8") as f:
            results.append(json.load(f))
    return results

get_previous_search_results()

[{'timestamp': '2025-11-12T08:18:00.115688+00:00',
  'query': 'Código de Minería Argentino servidumbres mineras relación con el dueño del suelo indemnización paso agua tierras concesionario artículos',
  'retrieved_documents': [{'index': 1,
    'distance': 0.7861493825912476,
    'tipo_parte': 'artículo',
    'numero_parte': 156,
    'titulo_parte': None,
    'ultimo_estado': 'en vigor',
    'contenido_texto': 'La concesión de una mina comprende el derecho el derecho de exigir la venta del terreno correspondiente.\n\nMientras tanto, se sujetará a lo dispuesto en el parágrafo de las servidumbres.',
    'codigo_norma': 'Código de Minería Argentino',
    'id_padre': 'cf24337039e4d80d81f68c9c9ea063bb',
    'id_parte': 'ab22d6e3cb64a7e29d0c0dc68298c74e'},
   {'index': 2,
    'distance': 0.7383683919906616,
    'tipo_parte': 'artículo',
    'numero_parte': 10,
    'titulo_parte': None,
    'ultimo_estado': 'en vigor',
    'contenido_texto': 'Sin perjuicio del dominio originario del Estado re

In [6]:
DATABASE_NAME = "tesis_MCD_v1"
COLLECTION_NAME = "codigo_de_mineria_v1"
EMBEDDING_MODEL_NAME = "hiiamsid/sentence_similarity_spanish_es"  # https://huggingface.co/hiiamsid/sentence_similarity_spanish_es/tree/main
ANSWER_VECTOR_FIELD_NAME = "vector_contenido_texto" # database field that contains the embedding vector to perform the search.
OUTPUT_FIELDS = [
    "codigo_norma",
    "id_parte",
    "id_padre",
    "tipo_parte",
    "numero_parte",
    "titulo_parte",
    "ultimo_estado",
    "contenido_texto",
]

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
client = MilvusClient(uri="http://localhost:19530", token="root:Milvus")
client.use_database(DATABASE_NAME)


question_vector = embedding_model.encode("Some Text")

results = client.search(
    collection_name=COLLECTION_NAME,
    anns_field=ANSWER_VECTOR_FIELD_NAME,
    data=[question_vector],
    limit=5,
    search_params={"metric_type": "COSINE"},
    output_fields=OUTPUT_FIELDS,
)